In [1]:
import duckdb

In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [12]:
df = con.execute("""
                SELECT * FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                    FROM bronze_z0019
                    WHERE data_ingestao >= '2026-07-07'
                ) WHERE row = 1
                """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,1001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-07 22:10:48.295697,1
1,1002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-07 22:10:48.295697,1
2,1004,SERRA,BT10,100,200,z0019_2.csv,2026-07-07 22:24:49.149001,1
3,1005,MACHADO,BT10,100,200,z0019_2.csv,2026-07-07 22:24:49.149001,1
4,1003,PREGO,BT10,100,60,z0019_2.csv,2026-07-07 22:24:49.149001,1


In [21]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={'NATBR': 'id'})
df_final = df_final.rename(columns={'MAKTX': 'nm_produto'})
df_final = df_final.rename(columns={'WERKS': 'id_categoria'})
df_final = df_final.rename(columns={'MAINS': 'id_fornecedor'})
df_final = df_final.rename(columns={'LABST': 'vl_preco'})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,1001,PARAFUSO,BT10,100,100
1,1002,MARTELO,BT50,100,1500
2,1004,SERRA,BT10,100,200
3,1005,MACHADO,BT10,100,200
4,1003,PREGO,BT10,100,60


In [24]:
df_final.dtypes

id               str
nm_produto       str
id_categoria     str
id_fornecedor    str
vl_preco         str
dtype: object

In [26]:
df2 = df_final
df2 = df2.astype(
    {
        'id': 'int64',
        'nm_produto': 'str',
        'id_categoria': 'str',
        'id_fornecedor': 'int64',
        'vl_preco': 'float64'
    }
)
df2.dtypes
#df2.head(10)

id                 int64
nm_produto           str
id_categoria         str
id_fornecedor      int64
vl_preco         float64
dtype: object

In [27]:
con.execute("""
            CREATE TABLE IF NOT EXISTS produtos (
                id BIGINT,
                nm_produto TEXT,
                id_categoria TEXT,
                id_fornecedor BIGINT,
                vl_preco FLOAT
            )
            """)

In [28]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,1001,PARAFUSO,BT10,100,100.0
1,1002,MARTELO,BT50,100,1500.0
2,1004,SERRA,BT10,100,200.0
3,1005,MACHADO,BT10,100,200.0
4,1003,PREGO,BT10,100,60.0


In [36]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [37]:
df_resultado = con.execute(" SELECT * FROM produtos ").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,1001,PARAFUSO,BT10,100,100.0
1,1002,MARTELO,BT50,100,1500.0
2,1004,SERRA,BT10,100,200.0
3,1005,MACHADO,BT10,100,200.0
4,1003,PREGO,BT10,100,60.0


In [38]:
con.close()